# 算子性能优化最佳实践

在前面的课程中，我们已经学习了算子核函数开发、Tiling设计和性能分析方法。完成算子的功能验证后，还需要根据性能数据判断：任务是否充分利用了多个核，计算是否在等待数据搬运，访存是否存在资源冲突。本节围绕这些问题，介绍 Ascend 芯片上的常见优化思路。

## 前置要求

学习本节前，建议掌握以下基础内容：

- 核函数的基本结构，以及通过Tiling划分数据的方式；
- GM与UB等存储层级，以及数据搬入、计算、搬出的基本流程；
- 使用性能分析工具观察算子耗时、流水利用率和访存情况的方法。

## 学习目标

完成本节学习后，能够：

- 理解多核并行与核间负载均衡对执行效率的影响；
- 说明Double Buffer与UnitFlag改善计算、搬运流水的基本思路；
- 了解数据拌匀、UB Bank冲突、L2 Cache控制和多核同地址冲突的优化方向；
- 根据性能瓶颈选择对应的最佳实践样例进行学习。

## 环境依赖

本节以原理介绍和样例阅读为主。如需编译运行链接中的样例，请选择`dav-2201`架构，并按对应README配置环境。

---

## 1. 多核任务划分

对于可以分块独立计算的算子，首先需要考虑如何将任务分配给多个核。这里有两个相关的问题：一是让多个核同时参与计算，二是让各核承担的工作量尽量接近。


### 1.1 多核并行

将可独立处理的数据块分配给多个计算核，让各核并行完成各自的任务。Tiling 需要确定核数、每核数据范围及尾块处理方式，同时保证输出范围正确且不重叠；核数应结合设备能力和任务规模选择。

具体实现可参考[Add 性能调优样例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/05_best_practices/00_vector_compute/add_high_performance)，对比 Case 1（单核向量）与 Case 2（多核均匀切分），关注核索引如何映射到数据起始位置。


### 1.2 核间负载均衡

多核任务的整体耗时容易受最慢的核影响。切分时应尽量让各核的计算量和搬运量接近，避免少数核承担过多尾块。对于每行工作量相同的任务，可以先平均分配行数，再把余数分散到前几个核，而不是全部交给最后一个核。

例如，将10行工作量相同的数据分给4个核，可以按`3、3、2、2`行分配，避免按`2、2、2、4`行分配后最后一个核明显更忙。实际切分还需要满足数据对齐和尾块处理要求。

具体实现可参考[Add 性能调优样例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/05_best_practices/00_vector_compute/add_high_performance)，重点查看 Case 2 中 `baseCoreM`、`remainderM` 和 `actualCoreM` 的计算，理解无法整除时如何均匀分配剩余行。


---

## 2. 计算与搬运流水优化

每个核内部同样存在并行机会。数据搬入、计算和结果搬出如果总是串行执行，计算单元就可能长时间等待。Double Buffer和UnitFlag分别从缓冲区交替使用、计算与结果搬出同步的角度改善流水。


### 2.1 Double Buffer

Double Buffer 使用两组缓冲区交替工作，使当前数据块的计算与其他数据块的搬运有机会重叠，减少流水等待。实现时需要配套同步，保证数据就绪后再计算、缓冲区释放后再复用；同时为两组缓冲区预留足够的片上空间。

例如，计算第一个数据块时，可以将下一个数据块搬入另一组缓冲区；下一轮交换两组缓冲区的角色。两组缓冲区需要正确同步，不能在计算尚未完成时覆盖其中的数据。

具体实现可参考[Add 性能调优样例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/05_best_practices/00_vector_compute/add_high_performance)，对比 Case 3 与 Case 4，关注 `ProcessDoubleBuffer()` 中 Ping/Pong 缓冲区及对应事件的切换。收益应以端到端耗时为准。


### 2.2 UnitFlag

在矩阵计算场景中，UnitFlag 可以让 MMAD 计算与 FIXPIPE 结果搬出采用更细粒度的同步，使计算和写回流水重叠，减少等待。若主要瓶颈在输入搬运侧，开启这一优化的收益可能有限。

具体实现可参考[Matmul 最佳实践样例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/05_best_practices/01_matrix_compute/matmul_high_performance)，对比 Case 7 与 Case 8，关注 `mmCFG.enUnitFlag = true` 的配置及 Cube/FIXPIPE 流水变化。


---

## 3. 访存优化

增加并行度后，数据供给仍可能成为瓶颈。此时需要进一步观察数据布局、片上访问冲突、缓存复用和多核访问顺序。下面四个方向分别从不同角度改善访存效率，应结合性能数据选择。


### 3.1 数据拌匀

数据拌匀关注数据地址分布与访存资源的利用：让访问尽量均匀地分散到存储资源上，减少访问集中造成的带宽瓶颈。设计数据布局和切分时，需要结合连续搬运长度、起始地址和步长，避免只看总数据量而忽略实际访问方式。

相关搬运实践可参考[DataCopy 最佳实践样例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/05_best_practices/04_memory_access/data_copy)。该样例没有单独命名为“数据拌匀”的场景，可结合“优化点1：分块粒度对搬运效率的影响”和“优化点2：非对齐数据搬运影响”观察布局与搬运方式对带宽的影响。样例对 Atlas A2/A3 的 half 数据建议主搬运维度对应的连续字节数满足 512B 对齐；这是该场景的调优建议，具体布局仍需实测。


### 3.2 UB Bank冲突

UB 由多个 Bank 和 Bank Group 组成。向量指令的多个操作数访问发生资源冲突时，会影响访问效率。可以结合输入、输出的 UB 地址和访问步长调整布局，减少冲突；是否冲突需按实际访问模式分析，不能仅凭两个 Tensor 地址不同就判断。

具体实现可参考[UB Bank 冲突及地址重叠场景分析示例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/05_best_practices/04_memory_access/bank_conflict_ub)，先对比 Scenario 1、2、3，观察源操作数地址布局与读读冲突的关系。再参考 [Add 性能调优样例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/05_best_practices/00_vector_compute/add_high_performance) 的 Case 6，了解实际算子中的 UB 布局优化。


### 3.3 L2 Cache控制

根据数据复用情况选择缓存策略。对于只读取一次、复用较少的流式输入，可以尝试 L2 Cache bypass；对于会重复使用的数据，应优先考虑调整分块与访问顺序，提高缓存命中机会。是否启用 bypass 要结合实际工作集及上下游访问情况验证。

具体实现可参考[Add 性能调优样例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/05_best_practices/00_vector_compute/add_high_performance) 的 Case 5，通过 `SetL2CacheHint(AscendC::CacheMode::CACHE_MODE_DISABLE)` 设置输入缓存策略；[Matmul 最佳实践样例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/05_best_practices/01_matrix_compute/matmul_high_performance) 的 Case 6 则展示通过分块促进 L2 Cache 复用的思路。


### 3.4 多核同地址冲突

多个核在同一时刻集中读取同一 GM 地址段，可能造成访存争用。对各核都需要读取的多块数据，可以按核索引错开访问起点或轮转分片顺序，使同一时刻的访问更分散，并保持每个核所需的数据覆盖完整。

具体实现可参考[DataCopy 最佳实践样例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/05_best_practices/04_memory_access/data_copy) 的“优化点4：多核同地址访问冲突规避”，对比 GM 到 UB 的 Scenario 7/8，或 GM 到 L1 的 Scenario 6/7，观察 `same addr` 与 `offset addr` 的差异。这里讨论的是并发读取的性能争用，错开读取顺序不能解决多个核写同一输出位置的正确性问题。


---

## 4. 本节小结

本节从多核、流水和访存三个方面介绍了算子性能优化的常见思路：

- 多核并行让多个核共同处理任务，负载均衡减少各核工作量差异带来的等待；
- Double Buffer利用两组缓冲区交替工作，UnitFlag改善矩阵计算与结果搬出的流水重叠；
- 数据布局、UB Bank冲突、L2 Cache策略和多核访问顺序都会影响访存效率。

实际调优时，应先定位瓶颈，再选择对应样例。每次调整一个因素，在相同输入和环境下验证结果正确性、比较端到端耗时，判断优化是否有效。
